# 2. Autopilot state machine

A mode state machine (`AP`: OFF -> ARMED -> ENGAGED) in `examples/autopilot/`:

* **Transition 2 has two rows** (switch still on *and* altitude captured). Rows that share a `transition_id` are ANDed: this is how guards are expressed.
* **Transition 3 is global** (empty `current_state`): disconnect returns to OFF from any state.
* A **monitoring equation** (`mon_no_cap`) runs after the state machine in the same band, so it sees the new state.

Everything runs at 20 Hz (50 ms per tick).

In [1]:
import sys, pathlib
try:
    import statesim
except ImportError:                      # not pip-installed: use the repo's src/ folder
    sys.path.insert(0, str(pathlib.Path("..").resolve() / "src"))
    import statesim
from statesim import load_model, Engine, ListTraceSink, FileTraceWriter, ModelError, EvaluationError
print("statesim imported")

statesim imported


In [2]:
model = load_model("../examples/autopilot/master.csv")
trace = ListTraceSink()
eng = Engine(model, trace=trace)


def show(label):
    s = eng.snapshot()
    print(f"{label:<34} tick={s['tick']:>2} t={s['t']:>6.1f} ms  AP={s['states']['AP']:<8} mon_no_cap={s['outputs']['mon_no_cap']}")


show("initial (before any step)")

initial (before any step)          tick= 0 t=   0.0 ms  AP=OFF      mon_no_cap=False


## Arm the autopilot

In [3]:
eng.set_input("ap_sw", True)
eng.step(); show("ap_sw = True")
eng.step(); show("next tick")

ap_sw = True                       tick= 1 t=   0.0 ms  AP=ARMED    mon_no_cap=False
next tick                          tick= 2 t=  50.0 ms  AP=ARMED    mon_no_cap=False


## Guard: engagement needs *both* rows true
The switch is still on, but `alt_cap` is False, so ARMED -> ENGAGED is blocked.

In [4]:
eng.run(3); show("waiting, no altitude capture")

waiting, no altitude capture       tick= 5 t= 200.0 ms  AP=ARMED    mon_no_cap=False


In [5]:
eng.set_input("alt_cap", True)
eng.step(); show("alt_cap = True")

alt_cap = True                     tick= 6 t= 250.0 ms  AP=ENGAGED  mon_no_cap=False


## Monitoring equation
`mon_no_cap` = *engaged without capture*. Dropping `alt_cap` while engaged raises it. (A state machine's state is a string variable named after its file: `AP == 'ENGAGED'`.)

In [6]:
eng.set_input("alt_cap", False)
eng.step(); show("alt_cap dropped while ENGAGED")

alt_cap dropped while ENGAGED      tick= 7 t= 300.0 ms  AP=ENGAGED  mon_no_cap=True


## Global transition
The disconnect transition has no departure state, so it works from any state.

In [7]:
eng.set_input("disc", True)
eng.step(); show("disc = True (global)")

disc = True (global)               tick= 8 t= 350.0 ms  AP=OFF      mon_no_cap=False


## Full trace

In [8]:
print("\n".join(trace.lines()))

tick=000000 t=0.000 | INPUT | set ap_sw False->True | ap_sw=True
tick=000001 t=0.000 | AP | t1_sw False->True | ap_sw=True
tick=000001 t=0.000 | AP | t2_sw False->True | ap_sw=True
tick=000001 t=0.000 | AP | T1 OFF->ARMED "Arm" | ap_sw=True
tick=000005 t=200.000 | INPUT | set alt_cap False->True | alt_cap=True
tick=000006 t=250.000 | AP | t2_cap False->True | alt_cap=True
tick=000006 t=250.000 | AP | T2 ARMED->ENGAGED "Engage" | alt_cap=True, ap_sw=True
tick=000006 t=250.000 | INPUT | set alt_cap True->False | alt_cap=False
tick=000007 t=300.000 | AP | t2_cap True->False | alt_cap=False
tick=000007 t=300.000 | EQ | mon_no_cap False->True | AP='ENGAGED', alt_cap=False
tick=000007 t=300.000 | INPUT | set disc False->True | disc=True
tick=000008 t=350.000 | AP | t3_disc False->True | disc=True
tick=000008 t=350.000 | AP | T3 ENGAGED->OFF "Disconnect" | disc=True
tick=000008 t=350.000 | EQ | mon_no_cap True->False | AP='OFF', alt_cap=False
